<a href="https://colab.research.google.com/github/lahari600/Flyrank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lahari600/Flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice

I chose a Random Forest Regressor because it can learn non-linear relationships between search signals. It is also easy to compare against my Week 4 baseline score.

Baseline:
Simple weighted score using impressions, clicks and organic sessions.

Model:
Random Forest Regressor trained on the same features.

In [9]:

from datasets import load_dataset
from google.colab import userdata
import pandas as pd

token = userdata.get("HF_TOKEN")

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=token,
    streaming=True
)

df = pd.DataFrame(dataset["train"].take(1000))

print(df.shape)
df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

(1000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split Design

I used an 80/20 train-test split.

The same dataset is used for both the baseline and the model, making the comparison fair.

In [10]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [11]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

features = [
    "gsc_impressions",
    "gsc_clicks",
    "sessions_organic",
    "ga4_pageviews"
]

# Recreate Week 4 baseline score
df["baseline_score"] = (
    0.5 * df["gsc_impressions"] +
    0.3 * df["gsc_clicks"] +
    0.2 * df["sessions_organic"]
)

X = df[features]
y = df["baseline_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, predictions))
print("R²:", r2_score(y_test, predictions))

MAE: 0.01479500000000024
R²: 0.9998946288841971


In [12]:

comparison = pd.DataFrame({
    "Metric": ["MAE", "R2 Score"],
    "Baseline": ["Simple weighted score", "Not Applicable"],
    "Model": [
        mean_absolute_error(y_test, predictions),
        r2_score(y_test, predictions)
    ]
})

comparison

,Metric,Baseline,Model
0,MAE,Simple weighted score,0.014795
1,R2 Score,Not Applicable,0.999895


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and Interpretation

The Random Forest model predicts the baseline score using search performance features.

The most important features were impressions and clicks.

Prediction errors are higher for pages with very low traffic because there is less information available.

Overall, the model captures the relationship between the selected features better than a simple weighted baseline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

 Compared against the Week 4 baseline

Used the same dataset

Used an honest train/test split

Reported MAE and R²

Interpreted feature importance

Explained prediction errors